# PakWheels - Model Training on Colab
Train a Random Forest model on the processed dataset.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

### Step 1: Connect Google Drive & Load Dataset
Make sure you upload `pakwheels_cars_processed.csv` to the `FYP_dataset` folder in your Google Drive before running this cell.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

# Using the FYP_dataset path in your Google Drive
INPUT_FILE = "/content/drive/MyDrive/FYP_dataset/pakwheels_cars_processed.csv"
df = pd.read_csv(INPUT_FILE)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,title,url,price_raw,body_type,assembly,exterior_color,registered_city,features,enrich_status,price,...,price_log,feature_count,body_type_encoded,assembly_encoded,exterior_color_encoded,registered_city_encoded,brand_clean,brand_encoded,model_clean,model_encoded
0,Honda N One 2014 Premium for Sale\n6.6/10,https://www.pakwheels.com/used-cars/honda-n-on...,PKR 25 lacs,Other,Other,Other,Other,NaN,Done,2500000.0,...,14.731802,0,0,0,0,0,Honda,5,Other,26
1,Toyota Tank 2019 for Sale\n7.1/10,https://www.pakwheels.com/used-cars/toyota-tan...,PKR 36 lacs,Other,Other,Other,Other,NaN,Pending,3600000.0,...,15.096445,0,0,0,0,0,Toyota,15,Other,26
2,KIA Sportage L 2025 HEV for Sale,https://www.pakwheels.com/used-cars/kia-sporta...,PKR 1.16 crore,Other,Other,Other,Other,NaN,Pending,11600000.0,...,16.266516,0,0,0,0,0,KIA,8,Other,26
3,Toyota Hilux 2017 Revo G 3.0 for Sale,https://www.pakwheels.com/used-cars/toyota-hil...,PKR 78.5 lacs,Other,Other,Other,Other,NaN,Pending,7850000.0,...,15.876024,0,0,0,0,0,Toyota,15,Hilux,20
4,Toyota Yaris Hatchback 2021 X for Sale,https://www.pakwheels.com/used-cars/toyota-yar...,PKR 46.95 lacs,Other,Other,Other,Other,NaN,Pending,4695000.0,...,15.362009,0,0,0,0,0,Toyota,15,Yaris Hatchback,39


### Step 2: Feature Selection & Data Prep

In [8]:
# Feature Selection
potential_features = [
    "year", "car_age", "mileage_km", "engine_cc", "feature_count",
    "fuel_type_encoded", "transmission_encoded", "body_type_encoded",
    "assembly_encoded", "registered_city_encoded", "exterior_color_encoded",
    "brand_encoded", "model_encoded"
]

# Only select features that actually exist in the uploaded CSV
features = [f for f in potential_features if f in df.columns]
target = "price"

# Drop rows with missing targets or features
df = df.dropna(subset=features + [target])

X = df[features]
y = df[target]

print(f"Training on {len(df)} records with {len(features)} features...")

Training on 8953 records with 7 features...


In [9]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 3: Train the Model

In [10]:
# Initialize and train the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


### Step 4: Evaluate Performance

In [11]:
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("[RESULTS]")
print(f"R-squared (Accuracy): {r2 * 100:.2f}%")
print(f"Mean Absolute Error:  PKR {mae:,.0f}")

# Print Feature Importances
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\nTop 5 Most Important Features for Pricing:")
for feat, imp in importances.head(5).items():
    print(f" - {feat}: {imp*100:.1f}%")

[RESULTS]
R-squared (Accuracy): 55.53%
Mean Absolute Error:  PKR 1,713,192

Top 5 Most Important Features for Pricing:
 - model_encoded: 80.5%
 - brand_encoded: 19.5%
 - feature_count: 0.0%
 - assembly_encoded: 0.0%
 - body_type_encoded: 0.0%


### Step 5: Save Model & Download

In [14]:
MODEL_OUTPUT = "/content/drive/MyDrive/FYP_dataset/car_price_model.pkl"
joblib.dump(model, MODEL_OUTPUT)

try:
    from google.colab import files
    files.download(MODEL_OUTPUT)
    print("Downloading model to your local machine...")
except ImportError:
    print(f"Model saved locally as {MODEL_OUTPUT}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>